# Suncroft / Staydia — train YOLOv8s + BotSort

**Before running:** Runtime → Change runtime type → **GPU**.

**Also needed:**
1. Roboflow project labelled (`player`, `ball`, `referee`) with a Version generated
2. Colab secrets: `ROBOFLOW_API_KEY`, `ROBOFLOW_WORKSPACE`, `ROBOFLOW_PROJECT`
3. For tracking: upload a short Suncroft mp4 to `/content/suncroft_clip.mp4`

Train with **YOLOv8-small** only (not newer YOLO). Roboflow = dataset; this notebook = real train.

In [ ]:
!pip -q install ultralytics roboflow

In [ ]:
import os
from roboflow import Roboflow

# Prefer Colab secrets; fall back to plain strings if you paste keys here.
try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
    WORKSPACE = userdata.get("ROBOFLOW_WORKSPACE")
    PROJECT = userdata.get("ROBOFLOW_PROJECT")
except Exception:
    API_KEY = "YOUR_ROBOFLOW_API_KEY"
    WORKSPACE = "daraghs-workspace-oiqly"
    PROJECT = "amateur-match-detection"

VERSION = 2  # detection with test1 frames

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("daraghs-workspace-oiqly").project("amateur-match-detection")
version = project.version(VERSION)
dataset = version.download("yolov8")
print("Dataset:", dataset.location)
print("data.yaml:", os.path.join(dataset.location, "data.yaml"))

In [ ]:
from ultralytics import YOLO
import os

data_yaml = os.path.join(dataset.location, "data.yaml")

# YOLOv8-small pretrained → fine-tune on Suncroft boxes
model = YOLO("yolov8s.pt")
model.train(
    data=data_yaml,
    epochs=50,
    imgsz=960,
    batch=16,   # lower to 8 if Colab runs out of GPU memory
    name="suncroft_v8s_960_v2",
)

BEST = "runs/detect/suncroft_v8s_960_v2/weights/best.pt"
print("best weights:", BEST)

In [ ]:
from ultralytics import YOLO
import os

BEST = "runs/detect/suncroft_v8s_960_v2/weights/best.pt"
model = YOLO(BEST)
metrics = model.val(
    data=os.path.join(dataset.location, "data.yaml"),
    imgsz=960,
)
print(metrics)

## BotSort on a clip

Upload a short Suncroft/Staydia mp4 in the left sidebar, then set `video_path` below (or rename the file to `/content/suncroft_clip.mp4`).

In [ ]:
from ultralytics import YOLO

BEST = "runs/detect/suncroft_v8s_960_v2/weights/best.pt"
video_path = "/content/suncroft_clip.mp4"

model = YOLO(BEST)
results = model.track(
    source=video_path,
    tracker="botsort.yaml",
    imgsz=960,
    conf=0.35,
    persist=True,
    save=True,
)
print("Tracking video is under runs/detect/track*/ — download it from the file browser.")

## After training

1. Download `runs/detect/suncroft_v8s_960_v2/weights/best.pt` to laptop `weights/` (gitignored).
2. Note mAP / class counts in weekly log.
3. When I add more Staydia labels, Generate a new Roboflow Version, bump `VERSION`, re-run train.